# Experiment 9a — The spike problem (Frontier 1)

First experiment of Phase 9. See [`../IMPROVEMENT-FRONTIERS.md`](../IMPROVEMENT-FRONTIERS.md)
for the full program. This notebook attacks **Frontier 1 — the spike tail**, the
highest-leverage unexplored pocket of error.

**The evidence that motivates this** (measured on the shipped blend OOF,
`experiment-7/oof_predictions_7b_blend.csv`):

| Finding | Number |
|---|---|
| Top 5% of rows by \|error\| | **36.2%** of the total WAPE numerator |
| Mean residual on top-5% highest-target rows | **-5.60** (severe under-prediction) |
| Mean residual overall | -0.485 |

The MAE objective finds the conditional **median**; on a right-skewed target the
model collapses toward the median and **cannot reach the peaks**. 36% of WAPE lives
in those peaks.

**Why experiment-7h ("bias correction hurts WAPE") does NOT close this door:** 7h
tested a *blanket additive shift*, which hurts the many already-correct rows. The
fix here is **conditional** - change the *fitting geometry* so the model represents
the tail better, not a uniform offset.

**Two honest caveats on the theory (stated up front, tested empirically):**

1. WAPE is scaled MAE, so its per-row optimum is the **conditional median** (q0.5) -
   the *same* optimum MAE already targets. Predicting a higher quantile (q0.65) is
   therefore **not** theoretically WAPE-optimal. We test it because the model
   *under-shoots* the tail in practice (a mis-specification / under-fit bias), and a
   higher quantile can compensate that bias on the spike rows. This is an empirical
   bet with a pre-registered test, not a guaranteed win.
2. The median is preserved under the monotonic `log1p`, so `log1p`-MAE also targets
   the median *in principle*. The expected benefit comes from (a) the changed loss
   geometry - large-value errors are compressed, so the tree fits bulk and tail more
   evenly - and (b) the **convex** `expm1` back-transform lifting averaged leaf
   predictions (Jensen). Again: tested, not assumed.

**Design choice - LGBM as the testbed.** Phase 9a is the *cheapest shot at the
biggest pocket*, so we run the treatments on the fast LightGBM anchor (CPU, minutes
for full 3-fold CV), exactly the anchor reproduced in 8a. If a treatment helps the
tree, it ports to the winning MLP in a follow-up (9a.1); if it fails even here,
that's a cheap negative. This keeps the notebook comparable to every prior
experiment.

**Treatment arms (all vs the LGBM-MAE anchor):**

| Arm | Objective | Target space | Targets the tail via |
|---|---|---|---|
| `mae` (anchor) | `regression_l1` | raw | - (baseline) |
| `logmae` | `regression_l1` | `log1p` -> `expm1` | loss geometry + convex back-transform |
| `pinball65` | `quantile`, alpha=0.65 | raw | asymmetric penalty on under-prediction |
| `logpinball65` | `quantile`, alpha=0.65 | `log1p` -> `expm1` | both |


---
## §0. Configuration

In [4]:
from pathlib import Path

DATA_DIR   = Path("/kaggle/input/datasets/xitij27/dlam-bonus-project-dataset/data")
OUTPUT_DIR = Path("/kaggle/working")
# Local:
# DATA_DIR   = Path("data")
# OUTPUT_DIR = Path("experiment-9")

TRAIN_PATH     = DATA_DIR / "train.csv"
VAL_INPUT_PATH = DATA_DIR / "validation_input.csv"

SEED         = 42
N_FOLDS      = 3
EVAL_HORIZON = 336

DOMAIN_FEAT_COLS = [
    "workload_intensity", "demand_forecast", "upstream_quality_forecast",
    "promotion_intensity", "shock_risk", "maintenance_known",
    "unit_reliability_forecast", "queue_pressure_forecast",
    "network_pressure_forecast", "event_load_forecast",
    "service_irregularity_risk_forecast",
]
TIME_COLS = ["hour_sin", "hour_cos", "dow_sin", "dow_cos",
             "doy_sin", "doy_cos", "is_weekend"]

BASELINE_LGBM = dict(
    n_estimators=600, learning_rate=0.03, num_leaves=63,
    subsample=0.8, colsample_bytree=0.8, min_child_samples=50,
    reg_lambda=1.0, verbose=-1,
)
TOP_DOM   = ["queue_pressure_forecast", "network_pressure_forecast",
             "workload_intensity", "shock_risk", "event_load_forecast"]
TOP3_ROLL = TOP_DOM[:3]
LAGS_DOM, LEADS_DOM, ROLL_WINS = [1, 24], [1, 24], [6, 24]

# -- Spike-experiment config ---------------------------------------------------
PINBALL_ALPHA = 0.65          # quantile for the asymmetric arms
SPIKE_PCT     = 0.05          # "spike rows" = top 5% of rows by true target
SEEDS         = [42, 101, 202]

# Decision thresholds -- pre-registered
ANCHOR_WAPE        = 14.78    # LGBM-MAE CV-WAPE (exp-4/5/6/8a)
ACCEPT_DELTA       = 0.05     # an arm must beat the anchor by > 0.05 WAPE overall
BLEND_R_THRESHOLD  = 0.85     # ensemble arm useful only if residual r < this

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"DATA_DIR   = {DATA_DIR}")
print(f"OUTPUT_DIR = {OUTPUT_DIR}")
print(f"Arms       : mae (anchor), logmae, pinball{int(PINBALL_ALPHA*100)}, "
      f"logpinball{int(PINBALL_ALPHA*100)}")
print(f"Spike defn : top {SPIKE_PCT*100:.0f}% of rows by true target")
print(f"Decisions  : accept arm if overall CV-WAPE < {ANCHOR_WAPE} - {ACCEPT_DELTA}; "
      f"blend arm if residual r < {BLEND_R_THRESHOLD}")


DATA_DIR   = /kaggle/input/datasets/xitij27/dlam-bonus-project-dataset/data
OUTPUT_DIR = /kaggle/working
Arms       : mae (anchor), logmae, pinball65, logpinball65
Spike defn : top 5% of rows by true target
Decisions  : accept arm if overall CV-WAPE < 14.78 - 0.05; blend arm if residual r < 0.85


---
## §1. Setup & data loading

In [5]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import lightgbm as lgb

np.random.seed(SEED)

def wape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    return np.sum(np.abs(y_true - y_pred)) / (np.sum(np.abs(y_true)) + eps) * 100

train_df = pd.read_csv(TRAIN_PATH, parse_dates=["timestamp"])
sid_to_code = {sid: i for i, sid in enumerate(sorted(train_df["series_id"].unique()))}
code_to_sid = {v: k for k, v in sid_to_code.items()}
N_SERIES = len(sid_to_code)
print(f"train_df: {train_df.shape}, # series: {N_SERIES}")
print(f"target range: [{train_df['target'].min():.3f}, {train_df['target'].max():.3f}]  "
      f"(log1p requires target >= -1 -- OK)")


train_df: (414720, 25), # series: 96
target range: [0.164, 53.000]  (log1p requires target >= -1 -- OK)


---
## §2. Feature engineering - same 52-feature tabular pipeline as 8a/exp-5.1

In [6]:
def impute_domain_features(df, cols):
    df = df.sort_values(["series_id", "timestamp"]).copy()
    df[cols] = df.groupby("series_id", group_keys=False)[cols].transform(
        lambda x: x.ffill().bfill()
    )
    return df

def add_time_features(df, time_col="timestamp"):
    df = df.copy()
    t = pd.to_datetime(df[time_col])
    df["hour_sin"]   = np.sin(2 * np.pi * t.dt.hour / 24)
    df["hour_cos"]   = np.cos(2 * np.pi * t.dt.hour / 24)
    df["dow_sin"]    = np.sin(2 * np.pi * t.dt.dayofweek / 7)
    df["dow_cos"]    = np.cos(2 * np.pi * t.dt.dayofweek / 7)
    df["doy_sin"]    = np.sin(2 * np.pi * t.dt.dayofyear / 365)
    df["doy_cos"]    = np.cos(2 * np.pi * t.dt.dayofyear / 365)
    df["is_weekend"] = (t.dt.dayofweek >= 5).astype(float)
    return df

def add_temporal_features(df, lag_cols, roll_cols, lags, leads, windows):
    df = df.copy().sort_values(["series_id", "timestamp"])
    new = []
    for col in lag_cols:
        g = df.groupby("series_id", sort=False)[col]
        for L in lags:
            c = f"{col}_lag{L}";  df[c] = g.shift(L);  new.append(c)
        for L in leads:
            c = f"{col}_lead{L}"; df[c] = g.shift(-L); new.append(c)
    for col in roll_cols:
        g = df.groupby("series_id", sort=False)[col]
        for W in windows:
            c_m = f"{col}_roll{W}m"
            df[c_m] = g.shift(1).transform(lambda x: x.rolling(W, min_periods=1).mean())
            new.append(c_m)
            c_s = f"{col}_roll{W}s"
            df[c_s] = g.shift(1).transform(lambda x: x.rolling(W, min_periods=1).std().fillna(0))
            new.append(c_s)
    return df, new

base_tab = add_time_features(impute_domain_features(train_df, DOMAIN_FEAT_COLS))
base_tab["sid_code"] = base_tab["series_id"].map(sid_to_code).astype("int32")
base_tab, TEMPORAL_FEATS = add_temporal_features(
    base_tab, lag_cols=TOP_DOM, roll_cols=TOP3_ROLL,
    lags=LAGS_DOM, leads=LEADS_DOM, windows=ROLL_WINS,
)
NUMERIC_FEATS  = DOMAIN_FEAT_COLS + TIME_COLS + ["series_mean"] + TEMPORAL_FEATS
FEATS_WITH_SID = NUMERIC_FEATS + ["sid_code"]
print(f"# features (incl. sid_code): {len(FEATS_WITH_SID)}")


# features (incl. sid_code): 52


---
## §3. CV framework + LGBM-MAE anchor

Same rolling-origin 3-fold split and anchor as 8a. `series_mean` is recomputed per fold from the fold's training slice only (no leakage).

In [7]:
def fold_offset(k):
    return (N_FOLDS - k) * EVAL_HORIZON

def fold_split_tabular(end_off, length=EVAL_HORIZON):
    tr_list, te_list = [], []
    for _, g in base_tab.groupby("series_id", sort=False):
        g = g.sort_values("timestamp")
        cut = len(g) - end_off
        tr_list.append(g.iloc[:cut])
        te_list.append(g.iloc[cut:cut + length])
    tr = pd.concat(tr_list, ignore_index=True)
    te = pd.concat(te_list, ignore_index=True)
    sm = tr.groupby("series_id")["target"].mean().rename("series_mean").reset_index()
    return tr.merge(sm, on="series_id"), te.merge(sm, on="series_id")

def fit_arm_predict(tr, te, feats, params, objective, *, alpha=None,
                    log_target=False, seeds=SEEDS, categorical=("sid_code",)):
    """Fit a seed-averaged LGBM arm with optional log1p target transform.

    log_target=True  -> fit on log1p(target), predict, expm1 back, clip at 0.
    objective='quantile' uses `alpha` (pinball). 'regression_l1' is MAE.
    """
    y_raw = tr["target"].to_numpy(float)
    y_fit = np.log1p(y_raw) if log_target else y_raw
    preds = []
    for s in seeds:
        kw = dict(**params, objective=objective, random_state=s)
        if alpha is not None:
            kw["alpha"] = alpha
        model = lgb.LGBMRegressor(**kw)
        model.fit(tr[feats], y_fit, categorical_feature=list(categorical))
        p = model.predict(te[feats])
        if log_target:
            p = np.expm1(p)
        preds.append(p)
    p = np.mean(preds, axis=0)
    return np.clip(p, 0.0, None)

ARMS = {
    "mae":          dict(objective="regression_l1", alpha=None, log_target=False),
    "logmae":       dict(objective="regression_l1", alpha=None, log_target=True),
    "pinball65":    dict(objective="quantile", alpha=PINBALL_ALPHA, log_target=False),
    "logpinball65": dict(objective="quantile", alpha=PINBALL_ALPHA, log_target=True),
}

print("Running 3-fold CV for all arms (seed-averaged)...")
oof_rows = []
for k in range(N_FOLDS):
    tr, te = fold_split_tabular(fold_offset(k))
    rec = te[["series_id", "timestamp", "target"]].copy()
    rec["fold"] = k
    rec["lead_h"] = rec.groupby("series_id").cumcount() + 1   # lead within the test block
    for name, cfg in ARMS.items():
        rec[f"pred_{name}"] = fit_arm_predict(
            tr, te, FEATS_WITH_SID, BASELINE_LGBM,
            cfg["objective"], alpha=cfg["alpha"], log_target=cfg["log_target"],
        )
    oof_rows.append(rec)
    wline = "  ".join(f"{name}={wape(rec['target'], rec[f'pred_{name}']):.2f}" for name in ARMS)
    print(f"  fold {k}: {wline}")

oof = pd.concat(oof_rows, ignore_index=True)
anchor_cv = np.mean([wape(g["target"], g["pred_mae"]) for _, g in oof.groupby("fold")])
assert abs(anchor_cv - ANCHOR_WAPE) < 0.5, f"anchor {anchor_cv:.3f} off expected {ANCHOR_WAPE}"
print(f"OK LGBM-MAE anchor reproduces: {anchor_cv:.3f} (~{ANCHOR_WAPE}).")


Running 3-fold CV for all arms (seed-averaged)...
  fold 0: mae=14.05  logmae=14.05  pinball65=13.97  logpinball65=14.01
  fold 1: mae=15.55  logmae=15.57  pinball65=16.05  logpinball65=16.03
  fold 2: mae=14.62  logmae=14.69  pinball65=15.03  logpinball65=15.36
OK LGBM-MAE anchor reproduces: 14.738 (~14.78).


---
## §4. Spike diagnostic - reproduce the Frontier-1 evidence on this CV

Confirm the concentration numbers from `IMPROVEMENT-FRONTIERS.md` hold on this harness's anchor OOF, and define the spike-row mask used to score every arm.

In [8]:
anchor_res = oof["target"] - oof["pred_mae"]
anchor_ae  = anchor_res.abs()
tot_num    = anchor_ae.sum()

print("=== Error concentration (LGBM-MAE anchor OOF) ===")
sorted_ae = np.sort(anchor_ae.to_numpy())[::-1]
N = len(oof)
for pct in [0.01, 0.05, 0.10]:
    k = int(N * pct)
    share = sorted_ae[:k].sum() / tot_num
    print(f"  top {pct*100:>2.0f}% rows by |err| = {100*share:4.1f}% of total WAPE numerator")

# spike rows = top SPIKE_PCT of rows by TRUE target
thresh = oof["target"].quantile(1 - SPIKE_PCT)
spike_mask = (oof["target"] >= thresh).to_numpy()
print(f"\nSpike rows = top {SPIKE_PCT*100:.0f}% by true target "
      f"(target >= {thresh:.2f}); n = {spike_mask.sum()} of {N}")
print(f"  mean residual overall          : {anchor_res.mean():+.3f}")
print(f"  mean residual on spike rows    : {anchor_res[spike_mask].mean():+.3f}  (anchor under-shoots)")
print(f"  spike rows' share of numerator : {100*anchor_ae[spike_mask].sum()/tot_num:.1f}%")


=== Error concentration (LGBM-MAE anchor OOF) ===
  top  1% rows by |err| =  9.9% of total WAPE numerator
  top  5% rows by |err| = 35.6% of total WAPE numerator
  top 10% rows by |err| = 51.6% of total WAPE numerator

Spike rows = top 5% by true target (target >= 21.47); n = 4839 of 96768
  mean residual overall          : +0.497
  mean residual on spike rows    : +5.836  (anchor under-shoots)
  spike rows' share of numerator : 19.2%


---
## §5. Evaluation - overall, spike-subset, full metric panel, residual r

For each arm: per-fold and overall CV-WAPE, WAPE **on the spike subset**, mean residual on spike rows (did we lift the peaks?), the full L2 panel for the Chronos question, and residual correlation vs the anchor (blend potential).

In [ ]:
def metrics_for(pred_col):
    p = oof[pred_col].to_numpy(); y = oof["target"].to_numpy()
    err = y - p
    per_fold = [wape(g["target"], g[pred_col]) for _, g in oof.groupby("fold")]
    res_anchor = (oof["target"] - oof["pred_mae"])
    res_arm    = (oof["target"] - oof[pred_col])
    r = float(np.corrcoef(res_anchor, res_arm)[0, 1])
    return dict(
        WAPE=float(np.mean(per_fold)),
        WAPE_std=float(np.std(per_fold)),
        WAPE_spike=wape(y[spike_mask], p[spike_mask]),
        resid_spike=float(err[spike_mask].mean()),
        MAE=float(np.mean(np.abs(err))),
        MSE=float(np.mean(err**2)),
        RMSE=float(np.sqrt(np.mean(err**2))),
        r_vs_anchor=r,
        per_fold=per_fold,
    )

results = {name: metrics_for(f"pred_{name}") for name in ARMS}
anchor_w = results["mae"]["WAPE"]

print(f"{'arm':<14} {'WAPE':>7} {'pm std':>7} {'WAPE_spk':>9} {'res_spk':>9} "
      f"{'MSE':>7} {'RMSE':>6} {'r_anc':>6}")
print("-" * 76)
for name in ARMS:
    m = results[name]
    flag = ""
    if name != "mae":
        d = anchor_w - m["WAPE"]
        flag = "  accept" if d > ACCEPT_DELTA else ("  ~noise" if d > -ACCEPT_DELTA else "  worse")
    print(f"{name:<14} {m['WAPE']:>7.3f} {m['WAPE_std']:>7.3f} {m['WAPE_spike']:>9.3f} "
          f"{m['resid_spike']:>+9.3f} {m['MSE']:>7.3f} {m['RMSE']:>6.3f} {m['r_vs_anchor']:>6.3f}{flag}")

print(f"\nAnchor spike residual was {results['mae']['resid_spike']:+.3f}; "
      "closer to 0 = peaks better captured.")
print("Reference: chronos OOF-equiv MSE ~7.72 (the L2 bar to beat for Overall rank).")


arm               WAPE  pm std  WAPE_spk   res_spk     MSE   RMSE  r_anc
----------------------------------------------------------------------------
mae             14.738   0.620    23.522    +5.836   8.854  2.976  1.000
logmae          14.769   0.623    23.147    +5.705   8.844  2.974  0.998  ~noise
pinball65       15.015   0.847    21.652    +5.181   8.630  2.938  0.995  worse
logpinball65    15.136   0.839    21.492    +5.072   8.695  2.949  0.993  worse

Anchor spike residual was +5.836; closer to 0 = peaks better captured.
Reference: chronos OOF-equiv MSE ~7.72 (the L2 bar to beat for Overall rank).


---
## §6. Blend exploration - does combining the anchor with the best arm help?

Same-family arms will be highly correlated, so a blend is unlikely to clear the 0.05 floor - but the spike-lifting arm may complement the anchor on the tail while the anchor protects the bulk. Grid the 2-way weight on OOF.

In [ ]:
best_arm = min((n for n in ARMS if n != "mae"), key=lambda n: results[n]["WAPE"])
print(f"Best single non-anchor arm by WAPE: {best_arm} ({results[best_arm]['WAPE']:.3f})")

pa = oof["pred_mae"].to_numpy()
pb = oof[f"pred_{best_arm}"].to_numpy()

def blend_cv_wape(w):
    oof["_tmp"] = w * pa + (1 - w) * pb
    pf = [wape(g["target"], g["_tmp"]) for _, g in oof.groupby("fold")]
    oof.drop(columns="_tmp", inplace=True)
    return float(np.mean(pf))

ws = np.round(np.arange(0.0, 1.001, 0.05), 3)
scan = [(w, blend_cv_wape(w)) for w in ws]
best_w, best_blend = min(scan, key=lambda t: t[1])
print(f"\n2-way blend  w*anchor + (1-w)*{best_arm}")
print(f"  w=1.00 (anchor only)   : {blend_cv_wape(1.0):.3f}")
print(f"  w=0.00 ({best_arm} only): {blend_cv_wape(0.0):.3f}")
print(f"  optimal w={best_w:.2f}         : {best_blend:.3f}")
print(f"  delta vs anchor: {anchor_w - best_blend:+.3f}  "
      f"({'clears' if anchor_w - best_blend > ACCEPT_DELTA else 'inside'} the {ACCEPT_DELTA} floor)")


Best single non-anchor arm by WAPE: logmae (14.769)

2-way blend  w*anchor + (1-w)*logmae
  w=1.00 (anchor only)   : 14.738
  w=0.00 (logmae only): 14.769
  optimal w=0.65         : 14.724
  delta vs anchor: +0.014  (inside the 0.05 floor)


---
## §7. Per-horizon WAPE

Kept for cohesion with 7a/8a. Does the spike treatment help uniformly across the horizon, or only at specific lead times?

In [ ]:
BINS = [(1, 1), (2, 6), (7, 24), (25, 72), (73, 168), (169, 336)]
cols = ["mae", best_arm]
hdr = "  ".join(f"{c:>10}" for c in cols)
print(f"{'lead':>10}  {'n':>6}  {hdr}")
print("-" * (24 + 12 * len(cols)))
for lo, hi in BINS:
    sl = oof[(oof["lead_h"] >= lo) & (oof["lead_h"] <= hi)]
    if len(sl) == 0: continue
    vals = "  ".join(f"{wape(sl['target'], sl[f'pred_{c}']):>10.3f}" for c in cols)
    print(f"  {lo:>3}..{hi:<3}h  {len(sl):>6}  {vals}")


      lead       n         mae      logmae
------------------------------------------------
    1..1  h     288      23.906      23.971
    2..6  h    1440      21.598      21.512
    7..24 h    5184      12.490      12.453
   25..72 h   13824      17.163      17.187
   73..168h   27648      14.012      13.976
  169..336h   48384      14.613      14.695


---
## §8. Save artifacts + decision summary

In [ ]:
oof_cols = ["series_id", "timestamp", "target", "lead_h", "fold"] + [f"pred_{n}" for n in ARMS]
oof_path = OUTPUT_DIR / "oof_predictions_9a_spike.csv"
oof[oof_cols].to_csv(oof_path, index=False)
print(f"OOF predictions -> {oof_path}")

summary = pd.DataFrame([{"arm": n, **{k: v for k, v in results[n].items() if k != "per_fold"}}
                        for n in ARMS])
summ_path = OUTPUT_DIR / "arm_summary_9a_spike.csv"
summary.to_csv(summ_path, index=False)
print(f"Arm summary -> {summ_path}")

print("\n=== Decision summary (Frontier 1 - spikes) ===")
print(f"  anchor (LGBM-MAE)      : WAPE {anchor_w:.3f}, spike-WAPE {results['mae']['WAPE_spike']:.3f}, "
      f"spike-resid {results['mae']['resid_spike']:+.3f}, MSE {results['mae']['MSE']:.3f}")
for name in ARMS:
    if name == "mae": continue
    m = results[name]
    print(f"  {name:<22}: WAPE {m['WAPE']:.3f} ({anchor_w - m['WAPE']:+.3f}), "
          f"spike-WAPE {m['WAPE_spike']:.3f}, spike-resid {m['resid_spike']:+.3f}, MSE {m['MSE']:.3f}")
print(f"  best 2-way blend       : WAPE {best_blend:.3f} ({anchor_w - best_blend:+.3f}) at w={best_w:.2f}")

any_accept = any((anchor_w - results[n]["WAPE"]) > ACCEPT_DELTA for n in ARMS if n != "mae") \
             or (anchor_w - best_blend) > ACCEPT_DELTA
print("\n=== Verdict ===")
if any_accept:
    print("ACCEPT: a spike treatment clears the 0.05 WAPE floor on the LGBM testbed.")
    print("   -> 9a.1: port the winning treatment to the gated-MLP (the 14.52 model) and re-evaluate.")
    print("   -> also report the MSE delta - a spike fix that also lowers MSE attacks the Chronos L2 gap.")
else:
    best_spike = min((n for n in ARMS if n != "mae"), key=lambda n: results[n]["WAPE_spike"])
    print("REJECT on LGBM: no treatment clears the WAPE floor.")
    print(f"   Best spike-subset WAPE: {best_spike} ({results[best_spike]['WAPE_spike']:.3f} "
          f"vs anchor {results['mae']['WAPE_spike']:.3f}).")
    print("   If spike-WAPE improved but overall didn't, the tail gain is offset by bulk loss")
    print("   on the tree - worth one MLP port (9a.1) before abandoning Frontier 1, since the")
    print("   MLP's smoother fit may bank the tail gain without the bulk regression.")


OOF predictions -> /kaggle/working/oof_predictions_9a_spike.csv
Arm summary -> /kaggle/working/arm_summary_9a_spike.csv

=== Decision summary (Frontier 1 - spikes) ===
  anchor (LGBM-MAE)      : WAPE 14.738, spike-WAPE 23.522, spike-resid +5.836, MSE 8.854
  logmae                : WAPE 14.769 (-0.031), spike-WAPE 23.147, spike-resid +5.705, MSE 8.844
  pinball65             : WAPE 15.015 (-0.277), spike-WAPE 21.652, spike-resid +5.181, MSE 8.630
  logpinball65          : WAPE 15.136 (-0.398), spike-WAPE 21.492, spike-resid +5.072, MSE 8.695
  best 2-way blend       : WAPE 14.724 (+0.014) at w=0.65

=== Verdict ===
REJECT on LGBM: no treatment clears the WAPE floor.
   Best spike-subset WAPE: logpinball65 (21.492 vs anchor 23.522).
   If spike-WAPE improved but overall didn't, the tail gain is offset by bulk loss
   on the tree - worth one MLP port (9a.1) before abandoning Frontier 1, since the
   MLP's smoother fit may bank the tail gain without the bulk regression.
